# Model #

I try to put down a model. I will use the xor-perceptron model as a base and modify that.

first create the lattice, N by N, with Nsol = N**2

I should create a class for the cell object and then the network class is formed by cell objects and when the network is created, each network is associated with a position in the lattice.

Note: for now I'm only considering one possible link between each pair neuron for each direction

In [1]:
import numpy as np
import jax
from jax import random as jrd
from jax import numpy as jnp
from jax import debug as jdb
from jax import lax
import graph_tool as gt
from graph_tool.all import *
import copy
from typing import NamedTuple
from functools import partial
import matplotlib.pyplot as plt


(ipykernel_launcher.py:50799): dbind-WARNING **: 11:35:57.869: AT-SPI: Error retrieving accessibility bus address: org.freedesktop.DBus.Error.ServiceUnknown: The name org.a11y.Bus was not provided by any .service files


In [2]:
# A QUANTO PARE IN REALTà QUESTA FUNZIONE NON MI SERVE, PERCHé VADO SEMPLICEMENTE A GENERARE TUTTE LE KEYS 
# CHE MI SERVONO NEL LOOP INIZIALE E POI LE DO IN INPUT AD OGNI FUNZIONE CHE NE NECESSITA
# define a quick function for generating a new random key from par['key'] and update par['key']
def gen_key(par:dict) -> tuple[jnp.ndarray,dict]:  # JAXXED!
    """Function to generate a new key, used for random generation in JAX, and update the current key
       present in par.

    Args:
        par (dict): dictionary of parameters, including also the current key.

    Returns:
        tuple[jax.KeyArray,dict]: [new key, updated par dictionary]
    """
    key, subkey = jrd.split(par['key'])     # generate a random key by splitting the key in par
    par['key'] = key                        # update the key in par
    return subkey, par                      # I have to return also par, otherwise, bc of JIT's rules, par['key'] won't update

class NetworkParams(NamedTuple):
    """NamedTuple class for the parameters of the network."""
    J: jnp.ndarray          # weight matrix
    C: jnp.ndarray          # connectivity matrix
    B: jnp.ndarray          # bias matrix
    G: Graph                # Graph of the network (Graph_tool)
    state: jnp.ndarray      # values of the cells in the network
    T: jnp.ndarray   # array of the T functions of the single cells in the network
    N: int = 0              # side lenght of the squared lattice (number of cells = N**2)
    fitness: float = -1.     # fitness value of the network
    
# ORA DEFINISCO LE VARIE FUNZIONI PER OPERARE SUI NETWORK
# - GENERATE(PAR,NETPAR): CREA (IN REALTà NON CREA L'OGGETTO) UN NETWORK RANDOM DATI I PARAMETRI 
#   PAR E RITORNA I PARAMETRI DEL NETWORK (NETPAR) AGGIORNATI
# - COMPUTE_FITNESS(PAR,NETPAR): CALCOLA LA FITNESS DEL NETWORK ATTRAVERSO IL METODO FF() E
#   RITORNA TALE FITNESS.
# - FF(INPUT(S),NETPAR): FA IL 'FEED-FORWARD' DEL NETWORK DATO UN SET DI INPUT E RITORNA 
#   LO STATE (OSSIA L'ARRAY DEGLI STATI 0 O 1 DELLE SINGOLE CELL)
    
def generate(subkeys:jnp.ndarray,par:dict) -> NetworkParams:  # JAXXED!
#def generate(self,subkeys:jnp.ndarray,N:int,p_self_link:float,N_Fourier:int) -> NetworkParams:  # JAXXED!
    """Generate a random network given the parameters par by updating the 
        NetPar object.

    Args:
        subkeys (jnp.ndarray): array of at least 4 keys for pseudorandom generation.
        par (dict): simulation/generation parameters.

    Returns:
        NetworkParams: returns the new network's parameters set. 
    """
    N = par['N']                                      
    J = jrd.uniform(subkeys[0],shape=(N**2,N**2))                 # uniformly populate the weights in the weight matrix J                                        # generate the subkey for the random generation in the following line
    B = jrd.normal(subkeys[1],shape=(N**2,N**2))                  # extract from a normal distribution centered in 0 with st. dv. = 1 the biases (spero vada bene fatto così)
    C = jnp.ones((N**2,N**2))       # as first, initialize C as fully connected (all ones)
    G = Graph(jnp.column_stack(jnp.nonzero(C)))         # Generate the Graph given the connectivity matrix
    state = jnp.abs(jrd.normal(subkeys[3],shape=(N**2,)))        # Randomly assign a state value to each cell distributed as the absolute value of a normal around 0
    f_n = par['T_initial'](par['x_n'])                  # evaluate the initial transfer function on x_n
    T = jnp.fft.fft(f_n)                                # compute the Fourier coefficients of f_n to move to the frequency space
    NetPar = NetworkParams(J,C,B,G,state,T,N)              # I don't specify the fitness argument so it stays as default (-1)
    fitness, state = compute_fitness(NetPar,par)                      # Compute the fitness value of the network (also updates the state)
    return NetworkParams(J,C,B,G,state,T,N,fitness)           # return the set of updated parameters

def compute_fitness(NetPar:NetworkParams,par:dict,verb:bool=False) -> tuple[float,jnp.ndarray]:    # JAXXED!
    """Function for computing the fitness of a given network.

    Args:
        NetPar (NetworkParams): parameter of the network in question.
        par (dict): simulation parameters.
        verb (bool, optional): verbosity. Defaults to False.

    Returns:
        tuple[float,jnp.ndarray]: computed fitness and updated states of the nodes of the network. 
    """
    fitness = 0.                # I need this check, bc otherwise I risk adding fitness over fitness  
    for i,input in enumerate(par['input_set']):
        output,state = ff(input,NetPar,par)
        if verb:
            jdb.print('Input: {input} -> {output}',input=input,output=output)
        norm_factor = NetPar.G.num_vertices()**2-NetPar.G.num_vertices()            # normalize by N(N-1)
        target_dist = (par['target_set'][i] - output)**2                            # square distance between network output and target (theoretical) output
        volume_cost = (jnp.sum(NetPar.C.reshape(-1))/norm_factor)**2                # average wiring volume cost (i.e. # of links)
        dist = shortest_distance(NetPar.G, directed=True).get_2d_array()            # calculate the shortest path length for each pair of vertecies
        dist = jnp.where(dist >= 2147483647, 0, dist)                               # set each value that overflows (bc no path exists) to 0
        length_cost = (jnp.sum((NetPar.C.reshape(-1) * dist.reshape(-1))
                                **par['length_powerlaw'])/norm_factor)**2            # average wiring length cost
        path_cost = (jnp.sum(dist,axis=[0,1])/norm_factor)**2                       # average shortest path length cost
        if verb:
            jdb.print('Target distance:{d}',d=target_dist)
            jdb.print('Volume cost:{d}',d=volume_cost)
            jdb.print('Length cost:{d}',d=length_cost)
            jdb.print('Path cost:{d}',d=path_cost)
        fitness += jnp.exp(target_dist + volume_cost + length_cost + path_cost) # take the exponential of the sum of the costs (weighted if needed)
    fitness /= len(par['input_set'])    
    return fitness, state
# then you call NetPar.fitness, NetPar.state = n.compute_fitness(...)

def ff(input:list,NetPar:NetworkParams,par:dict,verb:int=0) -> tuple[float,jnp.ndarray]:     # JAXXED!
    """Function to execute the 'feed-forward' computation on a given network, given inputs.

    Args:
        input (list): inputs to the network.
        NetPar (NetworkParams): network in question.
        par (dict): simulation parameters.
        verb (int, optional): verbosity. Defaults to 0.

    Returns:
        tuple[float,jnp.ndarray]: output of the 'feed-forward' and updated states of each node of the network.
    """
    state = NetPar.state                                # set the value of the two inputs cells through the input value
    state = state.at[0].set(input[0])                        # cell in the upper left corner of the 2D lattice
    state = state.at[(NetPar.N - 1) * NetPar.N-1].set(input[1])    # cell in the lower left corner of the 2D lattice
    # Precompute the contributions for each cell to optimize the loop (this is incredibly faster)
    contributions = jnp.dot(NetPar.C * NetPar.J, state) + jnp.sum(NetPar.B, axis=1)      # weight x value + bias - contributions così ha shape = state.shape
    # Retrive the T function of each cell by doing ifft and then fit/interpolation
    f_n = jnp.fft.ifft(NetPar.T,axis=1)      # inverse FT for moving back to coordinates space
    for cell in range(f_n.shape[0]):
        coeff = np.polyfit(par['x_n'],f_n[cell,:],5)    # fit the series of function value with a polynomial of degree up to 5
        new_fun = np.poly1d(coeff)                      # generate the new functional from the fit results
        state = state.at[cell].set(new_fun(contributions[cell]))    # apply the new functional as the T function for the associated cell
    if verb > 0: 
        jdb.print('State of each cell: {s}',s=state)
    output = state[-1]                          # the cell in the right lower corner is the output
    return output, state

In [ ]:
############# GENETIC ALGORITHM #################
def crossover(subkeys:jnp.ndarray,par1:NetworkParams,par2:NetworkParams) -> tuple[NetworkParams,NetworkParams]:
    """Function to generate 2 offsprings from 2 parents by crossover ricombination.

    Args:
        subkeys (jnp.ndarray): keys for random generation.
        par1 (NetworkParams): parent network #1.
        par2 (NetworkParams): parent network #2.

    Returns:
        tuple[NetworkParams,NetworkParams]: pair of offsprings.
    """   
    # Implements the crossover ricombination given 2 parent NetworkParams
    # and returns 2 offspring NetworksParams.
    if par1.C.shape != par2.C.shape:
        jdb.print('Error: length mismatch between the connectivity matrices of the two parents! Got parent1={p1} and parent2={p2}',
                  p1=par1.C.shape,p2=par2.C.shape)
        raise ValueError
    if par1.T.shape != par2.T.shape:
        jdb.print('Error: length mismatch between the T matrices of the two parents! Got parent1={p1} and parent2={p2}',
                  p1=par1.T.shape,p2=par2.T.shape)
        raise ValueError
    # First, I do C crossover
    cut_idx = jrd.choice(subkeys[0],jnp.arange(len(par1.C)))  # randomly pick where to cut the chromosomes
    C1 = jnp.append(par1.C.reshape(-1)[:cut_idx],par2.C.reshape(-1)[cut_idx:]).reshape(par1.C.shape)      # create the new C matrices by crossover
    C2 = jnp.append(par2.C.reshape(-1)[:cut_idx],par1.C.reshape(-1)[cut_idx:]).reshape(par1.C.shape)
    J1 = jnp.append(par1.J.reshape(-1)[:cut_idx],par2.J.reshape(-1)[cut_idx:]).reshape(par1.J.shape)      # create the new J matrices by crossover
    J2 = jnp.append(par2.J.reshape(-1)[:cut_idx],par1.J.reshape(-1)[cut_idx:]).reshape(par1.J.shape)
    J1 *= C1        # eliminate the weights for non existing links
    J2 *= C2
    B1 = jnp.append(par1.B.reshape(-1)[:cut_idx],par2.B.reshape(-1)[cut_idx:]).reshape(par1.B.shape)      # create the new B matrices by crossover
    B2 = jnp.append(par2.B.reshape(-1)[:cut_idx],par1.B.reshape(-1)[cut_idx:]).reshape(par1.B.shape)
    B1 *= C1        # eliminate the biases for non existing links
    B2 *= C2
    # Now T crossover
    cut_idx = jrd.choice(subkeys[1],jnp.arange(par1.T.shape[1]),(par1.T.shape[0],))  # randomly pick where to cut each cell's T coefficients string
    T1 = jnp.zeros((par1.T.shape))
    T2 = jnp.zeros((par1.T.shape))
    for cell in range(par1.T.shape[0]):      # I have to use this loop because append only works with one dimensional arrays
        T1 = T1.at[cell,:].set(jnp.append(par1.T[cell,:cut_idx[cell]],par2.T[cell,cut_idx:]))
        T2 = T2.at[cell,:].set(jnp.append(par2.T[cell,:cut_idx[cell]],par1.T[cell,cut_idx:]))
    # Now I set the other elements of NetPar that are not affected by the crossover
    state1 = jnp.abs(jrd.normal(subkeys[2],shape=par1.state.shape))     # State distributed as in the initialization because, biologically,
    state2 = jnp.abs(jrd.normal(subkeys[3],shape=par1.state.shape))     # it makes sense that the children do not inherit the value of the neurons of the parents
    G1 = Graph(jnp.column_stack(jnp.nonzero(C1)))                       # Generate the Graph given the connectivity matrix
    G2 = Graph(jnp.column_stack(jnp.nonzero(C2)))      
    # LE PROSSIME RIGHE SONO COMMENTATE PERCHé PER ORA NON MI SERVE CHE QUESTA FUNZIONE CALCOLI ANCHE LA FITNESS, PERCHé
    # LO FA LA FUNZIONE MUTATION, CHE PER ORA VIENE SEMPRE ESEGUITA UNA VOLTA CHE VIENE ESEGUITA QUESTA                 
    #NetPar1 = NetworkParams(J1,C1,B1,G1,state1,T1,par1.N)
    #NetPar2 = NetworkParams(J2,C2,B2,G2,state2,T2,par1.N)
    #fitness1, state1 = compute_fitness(par,NetPar1)
    #fitness2, state2 = compute_fitness(par,NetPar2)
    #return [NetworkParams(J1,C1,B1,G1,state1,T1,par1.N,fitness1), NetworkParams(J2,C2,B2,G2,state2,T2,par1.N,fitness2)]
    return (NetworkParams(J1,C1,B1,G1,state1,T1,par1.N), NetworkParams(J2,C2,B2,G2,state2,T2,par1.N))
    
def mutation(subkeys:jnp.ndarray,n:NetworkParams,par:dict) -> NetworkParams:
    """Function to mutate a given network.

    Args:
        subkeys (jnp.ndarray): keys for random generation.
        n (NetworkParams): parameters of the network to mutate.
        par (dict): simulation parameters.

    Returns:
        NetworkParams: mutated parameters of the network
    """
    # Function to mutate a given network
    ## Mutate C (probability of link inversly proportional to the length of the link)
    idx = jnp.arange(par['N']**2)
    row_idx = idx // par['N']
    col_idx = idx % par['N']
    row_diff = row_idx[:, None] - row_idx[None, :]
    col_diff = col_idx[:, None] - col_idx[None, :]
    dist_matrix = jnp.sqrt(row_diff**2 + col_diff**2)
    prob_matrix = jnp.where(jnp.eye(par['N']**2, dtype=bool),  # Create the probability matrix for connectivity
        par['p_self_link'],1/dist_matrix)
    mutationC = jnp.where(jrd.bernoulli(subkeys[0], jnp.array(prob_matrix)),1,0)    # Generate C   
    C_m = n.C * mutationC   # Apply the mutation
    ## Mutate J
    mutationJ = par['J_mutation_radius']*jrd.normal(subkeys[1],n.J.shape)   # mutation as gaussian noise (I multiply for the sd 
                                                                            # bc JAX only gives the unit normal)
    J_m = C_m * (n.J + mutationJ)        # Apply the mutation and eliminate weights for non existing links
    J_m = jnp.clip(J_m, min(par['J_range']), max(par['J_range']))   # Check the boundary conditions for J
    ## Mutate B
    mutationB = par['B_mutation_radius']*jrd.normal(subkeys[2],n.B.shape)   # mutation as gaussian noise (I multiply for the sd 
                                                                            # bc JAX only gives the unit normal)
    B_m = C_m * (n.B + mutationB)        # Apply the mutation and eliminate biases for non existing links
    # Mutate the T function
    mutationT = par['T_mutation_radius']*jrd.normal(subkeys[3],n.T.shape)   # mutation as gaussian noise (I multiply for the sd 
                                                                            # bc JAX only gives the unit normal)
    T_m = n.T + mutationT        # Apply the mutation 
    G_m = Graph(jnp.column_stack(jnp.nonzero(C_m)))     # create a new graph given the mutations
    NetPar = NetworkParams(J_m,C_m,B_m,G_m,n.state,T_m,n.N)
    fitness_m, state_m = compute_fitness(NetPar,par)
    return NetworkParams(J_m,C_m,B_m,G_m,state_m,T_m,n.N,fitness_m)

def evolution(subkeys:jnp.ndarray,par:dict,verb:int=1,early_stop:bool=True) -> tuple[list,list]:
    """Genetic algolrithm for evolving a network population (both the networks and the single nodes).

    Args:
        subkeys (jnp.ndarray): keys for random generation.
        par (dict): simulation parameters.
        verb (int, optional): verbosity. Defaults to 1.
        early_stop (bool, optional): early stopping condition. Defaults to True.

    Raises:
        ValueError: returns an error if par['N_sol'] is not even.
        ValueError: returns an error if the population is not conserved from one generation to the next.

    Returns:
        tuple[list,list]: offsprings list and mean fitness values list.
    """
    if par(['N_sol']) % 2 != 0:     # N_sol must be even
        jdb.print('Error:The number of solutions N_sol must be an even positive number.')
        raise ValueError
    solutions = []              # Generate the initial batch of solutions
    for n in range(par['N_sol']):
        # Generate a solution
        sol = generate(subkeys[:3],par)     # already computes also the fitness value
        solutions.append(sol)
    Fmean_values = []               # Initiate some container for statistic
    
    ##  EVOLUTION
    for iter in range(par['n_iter']):
        # generate the necessary keys
        n_parents = int(jnp.floor(par['N_sol'] * par['reproduction_ratio'])-jnp.floor(par['N_sol'] * par['reproduction_ratio'])%2)
        n_subkeys = (1+((n_parents/2)*(4+4+4))+((par['N_sol']-n_parents)*3)+1) + 1      # number of subkeys necessary in evolution (+1 for redefining par['key])
        subkeys = jrd.split(par['key'],n_subkeys)
        par['key'] = subkeys[-1]            # set the last key of subkeys as the new original key
        # Compute the mean fitness for statistics
        mean_fit = jnp.sum(jnp.array([sol.fitness for sol in solutions])) / par['N_sol']    # Here I don't have to divide also by 4, because I've already done it in the compute_fitness method
        # Save the old solution set for possible early stopping (QUESTO DOVREBBE TRIGGERARE SOLO SE SIAMO IN EARLY STOPPING IN QUESTA ITERAZIONE)
        '''
        if early_stop:
            solutions_old = []
            for sol in solutions:
                solutions_old.append(copy.deepcopy(sol))
        '''
        # SELECTION
        solutions = jnp.sort(solutions,key=lambda sol: sol.fitness)    # sort in ascending order based on fitness
        n_parents = int(jnp.floor(par['N_sol'] * par['reproduction_ratio'])-jnp.floor(par['N_sol'] * par['reproduction_ratio'])%2)  # the 2nd floor assures that n_parents is even
        parents_idx = jrd.choice(subkeys,jnp.arange(par['N_sol']),                       # must be with replacement, since the same individual 
                                 shape=(n_parents/2,2),replace=True,                 # can be chosen to be parent multiple times
                                 p=jnp.array([1/sol.fitness for sol in solutions])           # p: probability of being chosen as a parent, inversly proportional to the fitness
                                 /jnp.sum(jnp.array([sol.fitness for sol in solutions]))) 
        # REPRODUCTION
        offsprings = []
        for pair in range(len(parents_idx)):
            offspring = crossover(subkeys,solutions[parents_idx[pair,0]],solutions[parents_idx[pair,1]])    # generate 2 offspring by crossover
            offspring0 = mutation(subkeys,offspring[0],par)     # mutate them
            offspring1 = mutation(subkeys,offspring[1],par)
            offsprings.append([offspring0,offspring1])          # add them to the new population
        if iter%100 == 0 and verb == 1:
            print(f'Iteration #{iter}...')
            print(f'# of childs: {len(offspring)}')
        elif iter%10 == 0 and verb == 2:
            print(f'Iteration #{iter}...')
            print(f'# of childs: {len(offspring)}')
        # RANDOM GENERATION
        for _ in range(par['N_sol']-n_parents):
            sol = generate(subkeys,par)
            offsprings.append(sol)
        if len(offspring) != par['N_sol']:      # check population conservation
            jdb.print('Error: population not conserved; mismatching number of offsprings and parents. Got {o}, expected {p}',o=len(offsprings),p=par['N_sol'])
            raise ValueError
        offsprings = jrd.shuffle(subkeys,jnp.array(offsprings))     # shuffle the new population for good measure
        Fmean_values.append(mean_fit)   # statistics
    return offsprings, Fmean_values
        
########## UTILITY FUNCTIONS #################
def plot_T(T:jnp.ndarray,par:dict) -> None:
    f_n = jnp.fft.ifft(T,axis=1)
    coeff = np.polyfit(par['x_n'],f_n,5)    # fit the series of function value with a polynomial of degree up to 5
    new_fun = np.poly1d(coeff)                      # generate the new functional from the fit results
    x = jnp.arange(min(par['x_n']),max(par['x_n']),1000)
    plt.plot(x,new_fun(x))
    plt.show()
        
        
        

In [ ]:
# parameters of the simulation
par = {'key': jrd.key(1634),       # original key for random generation (this sets reproducibility)
     'N': 10,                    # side length of the squared lattice
     'int_range': 1,             # interaction range
     'p_self_link': 0.5,         # probability for each cell of having a self link
     'J_range': [-1,+1],                       # define the possible value for J.
     'J_mutation_radius': 0.1,                 # standard deviation of the normal distribution used to sample the mutation value for J.
     'B_mutation_radius': 0.1,            # standard deviation of the normal distribution used to sample the mutation value for B.
     'T_mutation_radius': 0.1,           # standard deviation of the normal distribution used to sample the mutation value for T.
     'reproduction_ratio': 0.8,          # percentage of offsprings to be generated by reproduction (the reamining by random generation)
     'length_powerlaw': 1,         # exponent of the wire length power law (DA DEFINIRE!!!)
     'x_n': jnp.arange(-10,10,200),   # discretization of the input of the transfer function (last argmument 
                                   # is also the maximum number of non-zero coefficients in the DFT of the 
                                   # transfer function. it should be a power of 2 for best performance of fft)
     'T_initial': lambda x: 0 + 1*x,              # initial transfer function shape (here linear)
     'target_set': [0.0,1.0,1.0,0.0],
     'input_set':[[0,0],[0,1],[1,0],[1,1]],
     'N_sol': 10}                       # number of solutions (individuals, networks) considered in the simulation

In [ ]:
#### EXECUTION
# Generate random keys
n_parents = int(jnp.floor(par['N_sol'] * par['reproduction_ratio'])-jnp.floor(par['N_sol'] * par['reproduction_ratio'])%2)
n_subkeys = 3 + par['n_iter'] * (1+((n_parents/2)*(4+4+4))+((par['N_sol']-n_parents)*3)+1) + 1      # number of subkeys necessary in evolution (+1 for redefining par['key])
subkeys = jrd.split(par['key'],n_subkeys)
par['key'] = subkeys[-1]            # set the last key of subkeys as the new original key

sol, Fmean = evolution(subkeys,par,verb=1)